# Customer support intent evaluation final analysis

Executed local reconciliation of the approved workbook, retained predictions and completed LangSmith annotation evidence. No cell makes a network request, model call or remote mutation. Keep the notebook beside submission_analysis.py and project.py. Failure reviews and annotation queues are complete. The participant still needs to record the Loom walkthrough; the optional LLM judge was not attempted.

In [1]:
import json
import project
import submission_analysis as analysis
final=analysis.reconcile()
print("Workbook and saved prediction evidence validated.")
print("Workbook SHA256:",final["workbook_sha256"])

Workbook and saved prediction evidence validated.
Workbook SHA256: 4bd5b5e4c3b9ad881bfa0333b4ecd13ed8a897397c9c10ce21ecd723f18f3103


## Dataset and exact match
The fixed upload has query, intent and id. Both fresh runs use gpt-4o-mini at temperature 0 and the same structured intent schema. Reference labels enter the evaluator only after classification.

In [2]:
dataset=analysis.read_csv(analysis.ROOT/'outputs/langsmith_dataset.csv')
assert len(dataset)==100 and len({r['id'] for r in dataset})==100
assert list(dataset[0])==['query','intent','id']
print('100 unique tickets; exact required columns.')
print(project.exact_match({'intent':'other'},{'intent':'other'}))
request=project.build_request('Where is my order?','baseline')
print('Model:',request['model'],'Temperature:',request['temperature'])

100 unique tickets; exact required columns.
{'key': 'intent_exact_match', 'score': 1}
Model: gpt-4o-mini Temperature: 0


## Single focused prompt change
Only the appended routing-precedence paragraph changed. The broader repository revision is separate reference evidence.

In [3]:
baseline=project.build_request('','baseline')['messages'][0]['content']
focused=project.build_request('','focused')['messages'][0]['content']
assert focused==baseline+'\n\n'+project.FOCUSED_ADDITION
print('BASELINE\n'+baseline)
print('\nONE ADDED PARAGRAPH\n'+project.FOCUSED_ADDITION)

BASELINE
You are a triage system for an e-commerce support inbox.
Classify the customer's ticket into EXACTLY ONE of these categories:
- order_status: questions about where an order is, tracking, delivery ETA
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)
Return only the intent key.

ONE ADDED PARAGRAPH
Routing precedence: classify the underlying problem before the requested remedy. When a refund is requested because an order never arrived, choose order_status; when it is requested because an item arrived damaged, wrong, defective, or not as described, choose product_issue. Otherwise keep the category definitions above unchanged.


## Original fresh experiment scores
These scores retain the original labels. The source repository separately reports 92/100 to 97/100 after a broader prompt revision; that does not measure this focused addition.

In [4]:
print(json.dumps(final['original_fresh_runs'],indent=2))
assert final['original_fresh_runs']['baseline']['correct']==92
assert final['original_fresh_runs']['focused']['correct']==94
original=analysis.read_csv(analysis.ROOT/'live_runs/comparison.csv')
print('Original wins:',sum(r['change']=='WIN' for r in original))
print('Original regressions:',sum(r['change']=='REGRESSION' for r in original))
print('Actual comparison:',final['langsmith_evidence']['comparison_url'])

{
  "baseline": {
    "correct": 92,
    "tickets": 100,
    "errors": 0
  },
  "focused": {
    "correct": 94,
    "tickets": 100,
    "errors": 0
  }
}
Original wins: 8
Original regressions: 6
Actual comparison: https://smith.langchain.com/o/b21f8ce6-e07f-4579-b2f2-b4ca372647f3/datasets/b97bfa75-95ea-4c09-b9b9-cbb8493da698/compare?selectedSessions=b2f408bc-391e-49a4-bb96-58ba78619308%2Cf1866797-b7bb-48c1-963a-2b154308d4fd


## Reviewed labels and final scores
The participant changed t046 from product_issue to refund_request on Fresh evaluation. Both unchanged predictions are rescored under that same label. Original LangSmith references remain intact; reviewed labels and scores are separate feedback fields.

In [5]:
print('Label changes:',json.dumps(final['label_changes'],indent=2))
m=final['reviewed_metrics']['Fresh evaluation']
print(json.dumps(m,indent=2))
assert (m['baseline_correct'],m['revised_correct'],m['wins'],m['regressions'])==(93,93,7,7)
assert m['delta_percentage_points']==0
print('Conclusion: no net accuracy gain; seven wins are offset by seven regressions.')

Label changes: [
  {
    "id": "t046",
    "source_intent": "product_issue",
    "reviewed_intent": "refund_request"
  }
]
{
  "tickets": 100,
  "baseline_correct": 93,
  "revised_correct": 93,
  "delta_percentage_points": 0.0,
  "wins": 7,
  "regressions": 7,
  "still_pass": 86,
  "still_fail": 0,
  "failed_cases": 14,
  "approved_failed_cases": 14,
  "pending_failed_cases": 0
}
Conclusion: no net accuracy gain; seven wins are offset by seven regressions.


## Failure analysis and regression checks
The baseline follows refund-remedy wording instead of underlying delivery/product routing. The focused change improves seven reviewed cases but regresses on six refund-payment/cancellation queries and t046, whose reviewed label changes the policy boundary.

In [6]:
fresh=final['workbook_rows']['Fresh evaluation']
for r in fresh:
    if r['change']=='REGRESSION':print(r['id'],'| reference:',r['reviewed_intent'],'| baseline:',r['baseline_prediction'],'| focused:',r['revised_prediction'])
assert next(r for r in fresh if r['id']=='t000')['change']=='STILL_PASS'
print('Previously passing regression control t000 remains PASS.')
print('Next test: distinguish a missing order from a missing refund payment; preserve cancellation/refund routing; settle the product-versus-remedy policy before testing unseen ticket families.')

t024 | reference: refund_request | baseline: refund_request | focused: order_status
t025 | reference: refund_request | baseline: refund_request | focused: order_status
t026 | reference: refund_request | baseline: refund_request | focused: order_status
t027 | reference: refund_request | baseline: refund_request | focused: order_status
t028 | reference: refund_request | baseline: refund_request | focused: order_status
t029 | reference: refund_request | baseline: refund_request | focused: order_status
t046 | reference: refund_request | baseline: refund_request | focused: product_issue
Previously passing regression control t000 remains PASS.
Next test: distinguish a missing order from a missing refund payment; preserve cancellation/refund routing; settle the product-versus-remedy policy before testing unseen ticket families.


## Completed workbook approvals
Column M marks review completion, not model correctness. A case failing either version is counted once per tab. The older Evaluation tab keeps the repository labels.

In [7]:
for name,m in final['reviewed_metrics'].items():
    print(name,':',m['approved_failed_cases'],'/',m['failed_cases'],'approved;',m['pending_failed_cases'],'pending')
    assert m['approved_failed_cases']==m['failed_cases'] and m['pending_failed_cases']==0
print('All 25 failed-case worksheet reviews are approved: 11 saved-run cases and 14 fresh cases.')

Evaluation : 11 / 11 approved; 0 pending
Fresh evaluation : 14 / 14 approved; 0 pending
All 25 failed-case worksheet reviews are approved: 11 saved-run cases and 14 fresh cases.


## Completed LangSmith annotation queues
Approved workbook reviews were imported through the API with provenance. Eight original baseline errors and seven focused errors under reviewed labels produce 15 run reviews across 14 tickets. t046 appears in both. All 75 feedback records were read back. For t025 and t026, blank column L entries use the pre-existing AI-assisted column K explanations, with that origin recorded. Repository-only Evaluation rows were not attached to unrelated live runs.

In [8]:
for q in final['queues']:
    print(q['name'],':',q['completed'],'/',q['expected'],'completed;',q['needs_my_review'],'pending')
    print(q['url'])
assert final['feedback_records_verified']==75
assert final['queue_run_reviews']==15 and final['unique_queue_tickets']==14
print('Verified feedback records:',final['feedback_records_verified'])
print('Column K fallback tickets:',final['notes_from_column_k'])
print('Original predictions and original evaluator feedback were preserved.')

Week4 P4 baseline approved Excel reviews : 8 / 8 completed; 0 pending
https://smith.langchain.com/o/b21f8ce6-e07f-4579-b2f2-b4ca372647f3/a/-/annotation-queues/9b8652c6-6d49-456a-b450-937684e6919f?runView=completed
Week4 P4 focused approved Excel reviews : 7 / 7 completed; 0 pending
https://smith.langchain.com/o/b21f8ce6-e07f-4579-b2f2-b4ca372647f3/a/-/annotation-queues/ac189235-b699-4e67-9cb8-f2d85ed9c34c?runView=completed
Verified feedback records: 75
Column K fallback tickets: ['t025', 't026']
Original predictions and original evaluator feedback were preserved.


## Approved note examples
Full notes and approvals are retained in the workbook and final review CSV exports. These examples include the participant-approved t046 routing-policy decision.

In [9]:
for ticket in ['t004','t024','t046']:
    r=next(r for r in fresh if r['id']==ticket)
    print('\n'+ticket+' | '+r['participant_approval'])
    print(r['participant_note'] or r['ai_draft_note'])


t004 | Approved
did not receive product. Issue refund

Draft explanation: The customer asks for a refund after non-delivery. The baseline follows the requested remedy and predicts refund_request. Under the current routing policy, the underlying missing order makes order_status correct; the revised prediction matches that label.

t024 | Approved
refund_request

t046 | Approved
Draft explanation: The reviewed label was changed to refund_request for this damaged-laptop ticket. The baseline refund_request therefore passes and the focused product_issue prediction fails. The source policy instead prioritizes product damage; confirm this reviewed routing choice before approval.


## Submission status and limits
Report, workbook, prompts, predictions, reviewed CSVs, queue mappings, verification evidence and screenshots are included. The remaining participant step is the Loom recording and its actual link. The optional judge was not run. These 100 tickets are related paraphrases of 25 seeds, not an unseen-family test. Failure approvals do not establish that all reference labels were manually audited.

Reproduce the analysis with `python finalize_submission.py`; no API calls are required. Paid classifier reruns are separate commands in project.py and are not executed by this notebook.

In [10]:
assert all(q['completed']==q['expected'] and q['needs_my_review']==0 for q in final['queues'])
print('Failure reviews: COMPLETE')
print('LangSmith annotation queues: COMPLETE')
print('Report and notebook reconciliation: COMPLETE')
print('Loom:',final['loom_status'])
print('Optional LLM judge:',final['optional_llm_judge'])

Failure reviews: COMPLETE
LangSmith annotation queues: COMPLETE
Report and notebook reconciliation: COMPLETE
Loom: Recording and actual link still required from participant
Optional LLM judge: Not attempted; optional
